# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates loading and processing the FAIR² dataset using the `mlcroissant` library. We will walk through the full pipeline: loading, overview, extraction, exploratory analysis, and basic visualization, referencing all dataset entities using their Croissant `@id` fields.

### Dataset Source
- Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and explore the global description with `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load dataset
dataset = mlc.Dataset(croissant_url)
print(f"Dataset Title: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(dataset.metadata, 'version', 'N/A')}")


## 2. Data Overview

Explore available record sets in the dataset Croissant metadata and list their `@id` fields, then show their fields and columns (all by `@id`).


In [ ]:
# List all record sets and their fields/columns by @id
if hasattr(dataset.metadata, 'recordSet'):
    record_sets = dataset.metadata.recordSet
    print(f"Record sets found: {len(record_sets)}")
    for i, rs in enumerate(record_sets):
        print(f"[{i}] RecordSet @id: {rs['@id']}")
        print(f"    Name: {rs.get('name','')}")
        # List fields for each record set
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        print(f"    Fields:")
        for field in fields:
            # Each field may be an @id reference or embedded
            if isinstance(field, dict) and '@id' in field:
                print(f"      - {field['@id']}")
            else:
                print(f"      - {field}")
        # List columns
        columns = rs.get('column', [])
        if isinstance(columns, dict):
            columns = [columns]
        print(f"    Columns:")
        for col in columns:
            if isinstance(col, dict) and '@id' in col:
                print(f"      - {col['@id']}")
            else:
                print(f"      - {col}")
        print()
else:
    print("No record sets found in metadata.")


## 3. Data Extraction

Load records from record sets into pandas DataFrames, referencing record sets and fields by their `@id`. This is the starting point for downstream tabular analysis.


In [ ]:
# Prepare a list of record set @ids (example: extract all or specify a subset)
record_set_ids = []
if hasattr(dataset.metadata, 'recordSet'):
    record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSet]

# Load all record sets into dataframes by @id
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns found: {df.columns.tolist()}")
    print(df.head(2), "\n")

# Use the first record set for demonstration
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id is not None:
    print(f"\nData sample from the main record set (`@id`: {main_record_set_id}):")
    display(dataframes[main_record_set_id].head())
else:
    print('No record set found in the dataset for extraction.')


## 4. Exploratory Data Analysis (EDA)

Now, let's pick a numeric field (referenced by its column `@id`) from the DataFrame and apply typical EDA steps: filtering by threshold, normalizing, and grouping by a categorical variable (all using `@id`).


In [ ]:
# Pick a numeric column `@id` available in the main DataFrame
if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Attempt automatic detection of a numeric field @id
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id:
        print(f"Detected numeric field @id: {numeric_field_id}")
        # Set filter threshold
        threshold = df[numeric_field_id].median() if not df[numeric_field_id].isnull().all() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where '{numeric_field_id}' > {threshold} (median):")
        print(filtered_df.head())

        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to find a suitable group field (categorical)
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            print(f"\nGrouping by field @id: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id, as_index=False)[numeric_field_id].mean().sort_values(numeric_field_id, ascending=False)
            print(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print("No numeric field detected in the main record set.")
else:
    print("Main record set DataFrame not loaded.")


## 5. Visualization

Visualize the normalized numeric field distribution and the grouped means if available, for `@id`-referenced fields.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[norm_col], bins=30, kde=True)
    plt.title(f"Normalized Distribution of '{numeric_field_id}' in Filtered Records")
    plt.xlabel(norm_col)
    plt.ylabel("Frequency")
    plt.show()
    # Grouped means barplot if grouping was performed
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(10,4))
        sns.barplot(
            data=grouped_df.head(10),
            x=group_field_id,
            y=numeric_field_id,
        )
        plt.title(f"Mean '{numeric_field_id}' by '{group_field_id}' (Top 10 groups)")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Not enough information for visualization (no numeric field detected).")


## 6. Conclusion

We successfully loaded and explored the FAIR² mass spectrometry dataset defined by a Croissant schema. Throughout, we referenced all core entities (record sets, fields/columns) by their `@id` identifiers per FAIR data best practices. Further domain-specific analysis can now be built on these standardized records and methods.
